In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pickle
import traceback
from pathlib import Path

from IPython.display import clear_output
from itertools import product
from sklearn.model_selection import train_test_split
from sklearn.impute import KNNImputer
from sklearn.metrics import r2_score
from matplotlib.colors import LogNorm
from pad_imputation import (
    convert_to_bool_coverage, load_wind_event, coverage_overlap, 
    load_random_file, intensity_histogram, solo, wind
)

In [ ]:
bin_width_deg = 4
time_avg_min = 5
avg_bin_dir = f"{time_avg_min}min_{bin_width_deg}deg"

intensity_path = Path("./data/intensities/") / avg_bin_dir
cov_path = Path("./data/coverages") / avg_bin_dir

In [ ]:
n_train = 75
n_test = 25

hists = []
reduced_hists = []

i = 0
j = 0
while i < n_train + n_test:
    hist = np.load(intensity_path)
    solo_cov = load_random_file("coverages/solo")
    try:
        reduced_hist = np.where(solo_cov, hist, np.nan)
        hists.append(hist)
        reduced_hists.append(reduced_hist)
        i += 1
    except ValueError:
        print(traceback.format_exc())
        j += 1
        continue
        
hists = np.vstack(hists)
reduced = np.vstack(reduced_hists)

train = hists[:n_train*720]
test = reduced[n_train*720:]
true = hists[n_train*720:]

#model = pickle.load(open('knnimputer_20251207', 'rb'))

In [ ]:
model = KNNImputer(n_neighbors=5, weights="distance")
model.fit(train)
dump_file = open('knnimputer_20251207_75_25', 'wb')
pickle.dump(model, dump_file)
dump_file.close()

In [ ]:
model = pickle.load(open("knnimputer_20251207_75_25", "rb"))

In [ ]:
test1 = model.transform(test[9*720:10*720])
true1 = true[9*720:10*720, :-1]    
reduced = test[9*720:10*720, :-1]
X1, Y1 = np.meshgrid(np.arange(0, test1.shape[0]), np.arange(0, test1.shape[1]), indexing="ij")
X2, Y2 = np.meshgrid(np.arange(0, true1.shape[0]), np.arange(0, true1.shape[1]), indexing="ij")
pred = np.where((np.isnan(reduced) & np.isfinite(true1)), test1, np.nan)
target = np.where(np.isfinite(pred), true1, np.nan)

In [ ]:
fig, axs = plt.subplots(nrows=5, figsize=(16,20))
hist_min = np.nanmin(np.vstack([test1, true1]))
hist_max = np.nanmax(np.vstack([test1, true1]))
axs[0].pcolormesh(X2, Y2, true1, norm=LogNorm(hist_min, hist_max), cmap="inferno")
axs[0].set_title("Wind intensities")
axs[1].pcolormesh(X2, Y2, reduced, norm=LogNorm(hist_min, hist_max), cmap="inferno")
axs[1].set_title("Wind intensities w/ reduction")
axs[2].pcolormesh(X2, Y2, test1, norm=LogNorm(hist_min, hist_max), cmap="inferno")
axs[2].set_title(f"k-NN imputed values, k = {5}")
axs[3].pcolormesh(X2, Y2, pred, norm=LogNorm(hist_min, hist_max), cmap="inferno")
axs[3].set_title("Predicted values")
axs[4].pcolormesh(X2, Y2, target, norm=LogNorm(hist_min, hist_max), cmap="inferno")
axs[4].set_title("Target values")

In [ ]:
target = target.flatten()
pred = pred.flatten()
r2_score(target[np.isfinite(target)], pred[np.isfinite(pred)])